# 20. Brand Portfolio Analysis

## Objective

This notebook examines the composition, structure and positioning of selected fragrance brand portfolios. The analysis focuses on manufacturers and their fragrance offerings rather than retailers or marketplace sellers.

The objective is to identify how selected luxury and niche brands differ in portfolio size, gender classification, fragrance accords, product concentration, fragrance families, notes and other product characteristics.

## Data Sources

Three sources are used:

- **Fragrantica:** Main source for fragrance catalogue records, brand names, gender classifications, main accords, community ratings and rating counts.
- **Scento:** Supplementary fragrance database providing concentration, launch year, fragrance families, character, seasons, times of day, notes and perfumer information.
- **10ML Fragrance Index API:** Additional source for individual fragrance profiles, including house, launch year, gender, family, accords, key notes and character.

## Brand Selection

The initial candidate sample contains 14 brands:

- **Luxury:** Dior, Guerlain, Chanel, Hermes, Mugler, Yves Saint Laurent and Tom Ford.
- **Niche:** Nicolai Parfumeur Createur, Nasomatto, Memo Paris, Byredo, 27 87, Maison Francis Kurkdjian and Serge Lutens.

The final sample will be determined after evaluating data availability across the sources.

## Analytical Approach

The analysis will examine:

1. Brand portfolio size and composition.
2. Gender classification and portfolio distribution.
3. Accord frequency, diversity and concentration.
4. Fragrance concentration and product characteristics.
5. Launch-year distribution and portfolio development.
6. Similarities and differences between individual brands.
7. Aggregate patterns across the selected luxury and niche groups.

## Methodological Considerations

The selected brands constitute a purposive sample and are not intended to represent the entire luxury or niche fragrance industry.

The datasets have different catalogue coverage and may contain overlapping fragrance records. Source-level counts will therefore be reported separately. Records will only be combined when fragrance identity can be established reliably.

The Fragrantica source table will remain unchanged. All transformations and analytical calculations will be performed in separate DataFrames.

In [0]:
%pip install perfume-api

In [0]:
# 1. Load data sources.

import requests
import pandas as pd
from io import StringIO
from pyspark.sql import functions as F

# Fragrantica
df_fragrantica = spark.table(
    "main.fragrance_analytics.fragrantica_perfumes"
)

print("Fragrantica perfumes:", df_fragrantica.count())

# Scento
SCENTO_BASE = (
    "https://huggingface.co/datasets/"
    "scento/perfume-database/resolve/main"
)

HEADERS = {
    "User-Agent": "FragranceRecommendationPlatform/0.1"
}

SOURCE_FILES = {
    "perfumes": f"{SCENTO_BASE}/perfumes.csv",
    "houses": f"{SCENTO_BASE}/houses.csv",
    "materials": f"{SCENTO_BASE}/materials.csv"
}

scento_data = {}

for name, url in SOURCE_FILES.items():
    response = requests.get(
        url,
        headers=HEADERS,
        timeout=30
    )
    response.raise_for_status()

    scento_data[name] = pd.read_csv(
        StringIO(response.text)
    )

    print(
        f"{name.upper()}: "
        f"{len(scento_data[name]):,} rows, "
        f"{len(scento_data[name].columns)} columns"
    )

In [0]:
# 2. Define candidate brands for portfolio analysis.

brand_groups = {
    "Luxury": [
        "Dior",
        "Guerlain",
        "Chanel",
        "Hermes",
        "Mugler",
        "Yves Saint Laurent",
        "Tom Ford"
    ],
    "Niche": [
        "Nicolai Parfumeur Createur",
        "Nasomatto",
        "Memo Paris",
        "Byredo",
        "Maison Francis Kurkdjian",
        "Serge Lutens"
    ]
}

brand_candidates = [
    (brand, group)
    for group, brands in brand_groups.items()
    for brand in brands
]

print(f"Candidate brands: {len(brand_candidates)}")

for group, brands in brand_groups.items():
    print(f"\n{group}: {len(brands)}")
    for brand in brands:
        print(f"  - {brand}")

In [0]:
# 3. Check candidate brand coverage in Fragrantica and Scento.

candidate_df = pd.DataFrame(
    brand_candidates,
    columns=["brand_name", "brand_group"]
)

candidate_df["brand_key"] = (
    candidate_df["brand_name"].str.strip().str.casefold()
)

# Fragrantica counts.
fragrantica_counts = (
    df_fragrantica
    .filter(F.col("brand_name").isNotNull())
    .withColumn(
        "brand_key",
        F.lower(F.trim(F.col("brand_name")))
    )
    .groupBy("brand_key")
    .agg(
        F.countDistinct("id").alias("fragrantica_count")
    )
    .toPandas()
)

# Scento counts.
df_scento = scento_data["perfumes"].copy()

df_scento["brand_key"] = (
    df_scento["house"]
    .astype("string")
    .str.strip()
    .str.casefold()
)

scento_counts = (
    df_scento
    .dropna(subset=["brand_key"])
    .groupby("brand_key")["id"]
    .nunique()
    .reset_index(name="scento_count")
)

# Combine coverage results.
brand_coverage = (
    candidate_df
    .merge(fragrantica_counts, on="brand_key", how="left")
    .merge(scento_counts, on="brand_key", how="left")
)

brand_coverage["fragrantica_count"] = (
    brand_coverage["fragrantica_count"].fillna(0).astype(int)
)

brand_coverage["scento_count"] = (
    brand_coverage["scento_count"].fillna(0).astype(int)
)

brand_coverage["both_datasets"] = (
    (brand_coverage["fragrantica_count"] > 0) &
    (brand_coverage["scento_count"] > 0)
)

brand_coverage["combined_source_records"] = (
    brand_coverage["fragrantica_count"] +
    brand_coverage["scento_count"]
)

brand_coverage = brand_coverage.sort_values(
    ["both_datasets", "combined_source_records"],
    ascending=[False, False]
).reset_index(drop=True)

display(
    brand_coverage[
        [
            "brand_name",
            "brand_group",
            "fragrantica_count",
            "scento_count",
            "both_datasets",
            "combined_source_records"
        ]
    ]
)

In [0]:
# 4. Inspect alternative brand and house names.

search_terms = [
    "chanel",
    "herm",
    "nicolai",
    "2787",
    "27 87",
    "serge lutens",
    "nasomatto"
]

print("FRAGRANTICA BRAND NAMES")

fragrantica_brands = (
    df_fragrantica
    .select("brand_name")
    .distinct()
    .filter(F.col("brand_name").isNotNull())
)

for term in search_terms:
    print(f"\nSearch: {term}")
    display(
        fragrantica_brands
        .filter(F.lower(F.col("brand_name")).contains(term))
        .orderBy("brand_name")
    )

print("\nSCENTO HOUSE NAMES")

scento_houses = sorted(
    scento_data["houses"]["name"].dropna().unique()
)

for term in search_terms:
    matches = [
        house for house in scento_houses
        if term in house.casefold()
    ]
    print(f"\nSearch: {term}")
    print(matches)

In [0]:
# 5. Explore candidate brands in the 10ML Fragrance Index.

from perfume_api import search_perfumes

api_search_results = {}
api_coverage_rows = []

for brand, group in brand_candidates:

    try:
        result = search_perfumes(
            brand,
            limit=5
        )

        api_search_results[brand] = result

        perfumes = result.get("results", [])

        api_coverage_rows.append({
            "brand_name": brand,
            "brand_group": group,
            "api_status": "OK",
            "api_search_total": result.get("total", 0),
            "returned_records": len(perfumes),
            "returned_houses": ", ".join(
                sorted({
                    perfume.get("house", "")
                    for perfume in perfumes
                    if perfume.get("house")
                })
            ),
            "returned_fragrances": ", ".join(
                perfume.get("name", "")
                for perfume in perfumes
            )
        })

    except Exception as e:

        api_search_results[brand] = None

        api_coverage_rows.append({
            "brand_name": brand,
            "brand_group": group,
            "api_status": str(e),
            "api_search_total": 0,
            "returned_records": 0,
            "returned_houses": "",
            "returned_fragrances": ""
        })

api_coverage = pd.DataFrame(api_coverage_rows)

display(api_coverage)

In [0]:
# 6. Inspect houses available in the 10ML Fragrance Index.

from perfume_api import list_houses

result = list_houses(limit=50)

print(result.keys())

api_houses = result["houses"]

df_api_houses = pd.DataFrame(api_houses)

display(df_api_houses.sort_values("name"))

In [0]:
# 7. Check candidate brand coverage in the 10ML house index.

api_brand_aliases = {
    "Hermes": "Hermès",
    "Nicolai Parfumeur Createur": "Nicolaï"
}

api_brand_coverage = candidate_df.copy()

api_brand_coverage["api_lookup_name"] = (
    api_brand_coverage["brand_name"]
    .replace(api_brand_aliases)
)

api_brand_coverage = api_brand_coverage.merge(
    df_api_houses[
        ["name", "perfumes_in_index"]
    ],
    left_on="api_lookup_name",
    right_on="name",
    how="left"
)

api_brand_coverage = api_brand_coverage.rename(
    columns={
        "name": "api_house_name",
        "perfumes_in_index": "api_perfume_count"
    }
)

display(
    api_brand_coverage[
        [
            "brand_name",
            "brand_group",
            "api_house_name",
            "api_perfume_count"
        ]
    ]
)

In [0]:
# 7. Check candidate brand coverage in the 10ML house index.

api_house_lookup = (
    df_api_houses
    .assign(house_key=lambda x: x["name"].str.casefold())
    .set_index("house_key")
)

api_brand_coverage = candidate_df.copy()

api_brand_coverage["house_key"] = (
    api_brand_coverage["brand_name"].str.casefold()
)

api_brand_coverage["api_house_name"] = (
    api_brand_coverage["house_key"]
    .map(api_house_lookup["name"])
)

api_brand_coverage["api_perfume_count"] = (
    api_brand_coverage["house_key"]
    .map(api_house_lookup["perfumes_in_index"])
)

display(
    api_brand_coverage[
        [
            "brand_name",
            "brand_group",
            "api_house_name",
            "api_perfume_count"
        ]
    ]
)

In [0]:
# 6. Fragrantica portfolio composition by brand and gender.

candidate_brand_names = [
    brand for brand, group in brand_candidates
]

df_fragrantica_candidates = (
    df_fragrantica
    .filter(
        F.col("brand_name").isin(candidate_brand_names)
    )
)

brand_gender_counts = (
    df_fragrantica_candidates
    .groupBy("brand_name", "gender")
    .agg(
        F.countDistinct("id").alias("fragrance_count")
    )
    .orderBy("brand_name", "gender")
)

display(brand_gender_counts)

In [0]:
%sql
DESCRIBE TABLE main.fragrance_analytics.fragrantica_perfumes;

In [0]:
%sql
WITH benchmarks AS (
    SELECT 'men' AS gender, 292.70 AS mean_attention, 4.000 AS mean_rating
    UNION ALL
    SELECT 'unisex', 141.28, 3.985
    UNION ALL
    SELECT 'women', 271.21, 3.968
),

guerlain AS (
    SELECT
        f.id,
        f.fragrance_name_clean AS fragrance,
        f.gender,
        f.rating_value AS rating,
        f.rating_count AS votes,
        f.main_accords
    FROM main.fragrance_analytics.fragrantica_perfumes f
    WHERE LOWER(f.brand_name) = 'guerlain'
      AND LOWER(f.gender) IN ('men', 'unisex', 'women')
      AND f.rating_value IS NOT NULL
      AND f.rating_count IS NOT NULL
)

SELECT
    g.fragrance,
    INITCAP(g.gender) AS gender,
    g.rating,
    g.votes,
    CASE
        WHEN g.rating >= b.mean_rating
         AND g.votes >= b.mean_attention
            THEN 'Standout Profiles'

        WHEN g.rating >= b.mean_rating
         AND g.votes < b.mean_attention
            THEN 'Hidden Favourites'

        WHEN g.rating < b.mean_rating
         AND g.votes >= b.mean_attention
            THEN 'Widely Explored'

        ELSE 'Quiet Profiles'
    END AS performance_segment,
    g.main_accords
FROM guerlain g
JOIN benchmarks b
    ON LOWER(g.gender) = b.gender
ORDER BY
    CASE LOWER(g.gender)
        WHEN 'men' THEN 1
        WHEN 'unisex' THEN 2
        WHEN 'women' THEN 3
    END,
    performance_segment,
    g.votes DESC;